# Proximity dose-response — distance sweep (seed 11)

The paper is titled around proximity but measures it at two points: the distance-preserving control
at ~101 km (+0.049) and the random rewire at ~511 km (+0.012). Two points cannot tell a smooth decay
from a threshold, and the random arm confounds distance with random assignment.

This sweep substitutes each true edge with a non-parent basin at a **target separation**, holding
in-degree exactly and excluding all true parents, so distance is the only thing that varies.
Dry-run-validated arms: 175 / 250 / 350 / 500 km, all with 0 of 624 edges overlapping the true graph.

Together with the existing ~101 km control and the ~511 km random rewire, this gives a five-point
curve from 100 to 500 km.

Pre-registration: `experiments/topology_ablation/preregistration_distance_sweep.md`.
**Falsification is live**: if Δ(500 km) ≈ Δ(101 km), distance is not the operative variable and the
paper's central claim reopens.

**Runtime → Change runtime type → T4 GPU → Run all.** ~3 h.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (SEED = 11)

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEED=11  # single seed: this buys the SHAPE of the curve, not per-point significance
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydro_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('CAMELS:', DRIVE_CAMELS_PATH); print('RUNS  :', DRIVE_RUNS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(f'{DRIVE_RUNS}/topology_ablation/component0', exist_ok=True)
print('datasets ->', os.path.realpath(RD)); print('runs     ->', os.path.realpath(RR))

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Build the swept-distance features

One feature per target distance. `--target-km X` matches every substitute edge to X km instead of to
its own true edge length. In-degree preserved, true parents excluded. Idempotent and guarded on a
`'date'`-named index.

In [ ]:
%cd {REPO_DIR}
import pickle
FEAT='experiments/topology_ablation/features'
TARGETS=[175,250,350,500]
def named_ok(p, n_expected=183):
    # must exist, have a 'date'-named index, AND cover every basin (NH KeyErrors on a missing one)
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True
for km in TARGETS:
    fp=f'{FEAT}/upstream_q_dist{km}km_component0_lag1.p'
    if named_ok(fp):
        print(f'{km} km: feature present, skipping'); continue
    print(f'=== building {km} km feature ===')
    !python experiments/topology_ablation/build_distance_control.py --network component0 --target-km {km} --lag-days 1
for km in TARGETS:
    fp=f'{FEAT}/upstream_q_dist{km}km_component0_lag1.p'
    print(f'  {km} km ->', named_ok(fp))

## Cell 8 — Train one condition per target distance

Four runs at seed 11. Idempotent per arm.

In [ ]:
%cd {REPO_DIR}
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
for km in TARGETS:
    cond=f'L_upQdist{km}km'
    if done(cond,SEED):
        print(f'{cond}: already done'); continue
    print(f'=== training {cond} seed {SEED} ===')
    !python experiments/topology_ablation/run_upstream_feature.py \
        --network component0 --seed {SEED} --device cuda:0 --epochs 30 \
        --feature-file experiments/topology_ablation/features/upstream_q_dist{km}km_component0_lag1.p \
        --cond-name {cond}
for km in TARGETS: print(f'  L_upQdist{km}km:', done(f'L_upQdist{km}km',SEED))

## Cell 9 — Verdict: the decay curve

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle, glob
from scipy.stats import wilcoxon
FEAT='experiments/topology_ablation/features'
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def named_ok(p, n_expected=183):
    # must exist, have a 'date'-named index, AND cover every basin (NH KeyErrors on a missing one)
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
_f=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
CONN=sorted([b for b,v in _f.items() if float(np.nanmax(np.abs(v.values)))>0])
def paired(cond,s,ref='L',basins=None):
    A=nse(cond,s); L=nse(ref,s)
    if A is None or L is None: return None
    bs=[b for b in (basins or CONN) if b in A.index and b in L.index]
    return (A[bs]-L[bs]).values
print('helpers ready | connected basins:', len(CONN))

In [ ]:
pts=[]
d=paired('L_upQ',SEED)
if d is not None: pts.append(('forward (true, ~92 km)', 92.0, np.median(d)))
d=paired('L_upQdistctrl',SEED)
if d is not None: pts.append(('distance-matched (~101 km)', 101.0, np.median(d)))
for km in TARGETS:
    d=paired(f'L_upQdist{km}km',SEED)
    if d is not None: pts.append((f'swept {km} km', float(km), np.median(d)))
d=paired('L_upQrand',SEED)
if d is not None: pts.append(('random rewire (~511 km)', 511.0, np.median(d)))

print('| condition | mean edge km | paired median ΔNSE |')
print('|---|---|---|')
for n,k,v in pts: print(f'| {n} | {k:.0f} | {v:+.4f} |')

sweep=[(k,v) for n,k,v in pts if 'swept' in n or 'distance-matched' in n]
if len(sweep)>=4:
    ks=np.array([k for k,_ in sweep]); vs=np.array([v for _,v in sweep])
    from scipy.stats import spearmanr
    rho,p=spearmanr(ks,vs)
    near=vs[0]; far=vs[-1]
    print(f'\nSpearman(target distance, Δ) = {rho:+.3f} (p={p:.3f})')
    print(f'near ({ks[0]:.0f} km) Δ={near:+.4f}   far ({ks[-1]:.0f} km) Δ={far:+.4f}   drop={near-far:+.4f}')
    print('\n=== PRE-REGISTERED VERDICT ===')
    if rho<0 and (near-far)>0.005:
        mono=all(vs[i]>=vs[i+1]-0.005 for i in range(len(vs)-1))
        print(f'  DOSE-RESPONSE CONFIRMED (monotone within noise: {mono}).')
        print('  Proximity is the operative variable; report the decay curve and the length scale.')
    elif abs(near-far)<=0.005:
        print('  FALSIFIED: distance does NOT drive the gain (far ~= near).')
        print('  The random rewire failed for another reason. The proximity framing must be reopened.')
    else:
        print('  MIXED / non-monotone — inspect the table before writing.')
else:
    print('\nnot enough arms completed to judge')

## Cell 10 — Persistence check

In [ ]:
print('=== persistence (in Drive?) ===')
for km in TARGETS:
    dp=f'{DRIVE_RUNS}/topology_ablation/component0/L_upQdist{km}km_component0_seed{SEED}/test/model_epoch030/test_metrics.csv'
    print(f'  L_upQdist{km}km: {os.path.isfile(dp)}')

## Done

Four runs persist to Drive. Report the **Cell 9 table + verdict** and **Cell 10** back.

If the curve is clean this replaces the paper's argument-by-elimination with a measured length scale,
and it is the direct answer to "your title's variable is measured at two levels."